# Unidad 5: Transformaciones avanzadas

**Tecnicatura en Datos – Procesamiento con Apache Spark (Databricks)**  
Unidad 5 de 10 — Duración estimada: 2:30 hs

> **Nota Databricks:** `spark` ya está disponible. Todos los ejemplos usan `createDataFrame` para ser portables. En producción, reemplazar por `spark.read.format(...)`.

In [ ]:
from pyspark.sql.functions import (
    sum, avg, count, max, min, col,
    rank, dense_rank, row_number,
    lag, lead, sum as spark_sum,
    when, udf
)
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
print("Imports OK | Spark", spark.version)

---
## 1. Agregaciones

### Ejemplo 1 — Simple: `groupBy` + `agg` por país

In [ ]:
data = [
    ("AR", "laptop",  1200.0),
    ("AR", "mouse",     80.0),
    ("MX", "teclado",  150.0),
    ("MX", "monitor",  400.0),
    ("AR", "webcam",   120.0),
    ("CL", "laptop",  1100.0),
]
df = spark.createDataFrame(data, ["pais", "producto", "monto"])

resumen = (
    df
    .groupBy("pais")
    .agg(
        sum("monto").alias("monto_total"),
        avg("monto").alias("monto_promedio"),
        count("producto").alias("cant_operaciones"),
    )
    .orderBy("monto_total", ascending=False)
)
resumen.show()

**Resultado esperado:**
```
+----+-----------+------------------+----------------+
|pais|monto_total|    monto_promedio|cant_operaciones|
+----+-----------+------------------+----------------+
|  AR|     1400.0|466.66666666666667|               3|
|  CL|     1100.0|            1100.0|               1|
|  MX|      550.0|             275.0|               2|
+----+-----------+------------------+----------------+
```

### Ejemplo 2 — Medio: múltiples métricas con filtro post-agg

In [ ]:
data_tx = [
    (1, "Ana",   "AR", 1500.0),
    (2, "Luis",  "MX",  800.0),
    (3, "Ana",   "AR",  200.0),
    (4, "Marta", "AR", 2200.0),
    (5, "Luis",  "MX",  300.0),
    (6, "Pedro", "CL",  450.0),
]
df_tx = spark.createDataFrame(data_tx, ["id", "cliente", "pais", "monto"])

metrics = (
    df_tx
    .groupBy("cliente", "pais")
    .agg(
        sum("monto").alias("total"),
        count("id").alias("operaciones"),
        max("monto").alias("monto_max"),
        min("monto").alias("monto_min"),
    )
    .filter((col("operaciones") > 1) & (col("total") > 500))
    .orderBy("total", ascending=False)
)
metrics.show()

**Resultado esperado:**
```
+-------+----+------+-----------+---------+---------+
|cliente|pais| total|operaciones|monto_max|monto_min|
+-------+----+------+-----------+---------+---------+
|    Ana|  AR|1700.0|          2|   1500.0|    200.0|
|   Luis|  MX|1100.0|          2|    800.0|    300.0|
+-------+----+------+-----------+---------+---------+
```

### Ejemplo 3 — Avanzado: `pivot` para convertir filas en columnas

In [ ]:
data_pivot = [
    ("enero",   "AR", 3000.0),
    ("enero",   "MX", 1500.0),
    ("enero",   "CL",  800.0),
    ("febrero", "AR", 4200.0),
    ("febrero", "MX", 2100.0),
    ("febrero", "CL", 1100.0),
]
df_pivot = spark.createDataFrame(data_pivot, ["mes", "pais", "ventas"])

tabla = (
    df_pivot
    .groupBy("mes")
    .pivot("pais", ["AR", "CL", "MX"])   # lista explícita = más eficiente
    .agg(sum("ventas"))
    .orderBy("mes")
)
tabla.show()

**Resultado esperado:**
```
+-------+------+------+------+
|    mes|    AR|    CL|    MX|
+-------+------+------+------+
|  enero|3000.0| 800.0|1500.0|
|febrero|4200.0|1100.0|2100.0|
+-------+------+------+------+
```

---
## 2. Joins

### Ejemplo 1 — Simple: `inner join`

In [ ]:
transacciones = spark.createDataFrame([
    (101, 1, 1500.0),
    (102, 2,  800.0),
    (103, 1, 2200.0),
    (104, 3,  450.0),
], ["tx_id", "cliente_id", "monto"])

clientes = spark.createDataFrame([
    (1, "Ana",   "AR"),
    (2, "Luis",  "MX"),
    (3, "Marta", "AR"),
], ["cliente_id", "nombre", "pais"])

resultado = transacciones.join(clientes, "cliente_id", "inner")
resultado.show()

**Resultado esperado:**
```
+----------+------+------+------+----+
|cliente_id| tx_id| monto|nombre|pais|
+----------+------+------+------+----+
|         1|   101|1500.0|   Ana|  AR|
|         1|   103|2200.0|   Ana|  AR|
|         2|   102| 800.0|  Luis|  MX|
|         3|   104| 450.0| Marta|  AR|
+----------+------+------+------+----+
```

### Ejemplo 2 — Medio: `left join` para detectar huérfanos

In [ ]:
transacciones_ext = spark.createDataFrame([
    (101, 1, 1500.0),
    (102, 2,  800.0),
    (103, 99, 2200.0),   # cliente 99 no existe
    (104, 3,  450.0),
], ["tx_id", "cliente_id", "monto"])

resultado = (
    transacciones_ext
    .join(clientes, "cliente_id", "left")
    .select("tx_id", "cliente_id", "monto", "nombre", "pais")
)
resultado.show()

print("Transacciones SIN cliente (huérfanos):")
resultado.filter(col("nombre").isNull()).show()

**Resultado esperado:**
```
+------+----------+------+------+----+
| tx_id|cliente_id| monto|nombre|pais|
+------+----------+------+------+----+
|   101|         1|1500.0|   Ana|  AR|
|   102|         2| 800.0|  Luis|  MX|
|   103|        99|2200.0|  null|null|   ← huérfano
|   104|         3| 450.0| Marta|  AR|
+------+----------+------+------+----+
```

### Ejemplo 3 — Avanzado: join con condición compuesta y límite de crédito

In [ ]:
limites = spark.createDataFrame([
    (1, "AR", 2000.0),
    (2, "MX", 1000.0),
    (3, "AR", 1500.0),
], ["cliente_id", "pais", "limite_credito"])

df_tx_pais = spark.createDataFrame([
    (101, 1, "AR", 1500.0),
    (102, 2, "MX",  800.0),
    (103, 1, "AR", 2500.0),   # supera límite
    (104, 3, "AR",  450.0),
    (105, 2, "MX", 1200.0),   # supera límite
], ["tx_id", "cliente_id", "pais", "monto"])

resultado = (
    df_tx_pais
    .join(limites, on=["cliente_id", "pais"], how="inner")
    .withColumn("supera_limite",
        when(col("monto") > col("limite_credito"), "SÍ").otherwise("NO")
    )
    .select("tx_id", "cliente_id", "pais", "monto", "limite_credito", "supera_limite")
    .orderBy("tx_id")
)
resultado.show()

**Resultado esperado:**
```
+------+----------+----+------+--------------+-------------+
| tx_id|cliente_id|pais| monto|limite_credito|supera_limite|
+------+----------+----+------+--------------+-------------+
|   101|         1|  AR|1500.0|        2000.0|           NO|
|   102|         2|  MX| 800.0|        1000.0|           NO|
|   103|         1|  AR|2500.0|        2000.0|           SÍ|
|   104|         3|  AR| 450.0|        1500.0|           NO|
|   105|         2|  MX|1200.0|        1000.0|           SÍ|
+------+----------+----+------+--------------+-------------+
```

---
## 3. Window Functions

### Ejemplo 1 — Simple: `row_number` por cliente

In [ ]:
data = [
    (1, "Ana",  "2024-01-05", 1500.0),
    (2, "Ana",  "2024-01-10",  200.0),
    (3, "Luis", "2024-01-03",  800.0),
    (4, "Ana",  "2024-01-15", 2200.0),
    (5, "Luis", "2024-01-08",  300.0),
]
df = spark.createDataFrame(data, ["id", "cliente", "fecha", "monto"])

ventana = Window.partitionBy("cliente").orderBy("fecha")

df.withColumn("orden", row_number().over(ventana)).show()

**Resultado esperado:**
```
+---+-------+----------+------+-----+
| id|cliente|     fecha| monto|orden|
+---+-------+----------+------+-----+
|  1|    Ana|2024-01-05|1500.0|    1|
|  2|    Ana|2024-01-10| 200.0|    2|
|  4|    Ana|2024-01-15|2200.0|    3|
|  3|   Luis|2024-01-03| 800.0|    1|
|  5|   Luis|2024-01-08| 300.0|    2|
+---+-------+----------+------+-----+
```

### Ejemplo 2 — Medio: `rank` vs `dense_rank` con empates

In [ ]:
data_rank = [
    ("Ana",  1500.0),
    ("Ana",  1500.0),   # empate
    ("Ana",   200.0),
    ("Luis",  800.0),
    ("Luis",  800.0),   # empate
    ("Luis",  300.0),
]
df_r = spark.createDataFrame(data_rank, ["cliente", "monto"])

ventana_monto = Window.partitionBy("cliente").orderBy(col("monto").desc())

(
    df_r
    .withColumn("rank",       rank().over(ventana_monto))
    .withColumn("dense_rank", dense_rank().over(ventana_monto))
    .show()
)

**Resultado esperado:**
```
+-------+------+----+----------+
|cliente| monto|rank|dense_rank|
+-------+------+----+----------+
|    Ana|1500.0|   1|         1|
|    Ana|1500.0|   1|         1|
|    Ana| 200.0|   3|         2|  ← rank salta a 3; dense_rank es 2
|   Luis| 800.0|   1|         1|
|   Luis| 800.0|   1|         1|
|   Luis| 300.0|   3|         2|
+-------+------+----+----------+
```

### Ejemplo 3 — Avanzado: `lag`, `lead` y suma acumulada

In [ ]:
data_tend = [
    ("Ana", "2024-01-05",  500.0),
    ("Ana", "2024-01-10", 1500.0),
    ("Ana", "2024-01-15",  200.0),
    ("Ana", "2024-01-20", 2200.0),
]
df_tend = spark.createDataFrame(data_tend, ["cliente", "fecha", "monto"])

ventana_hist = Window.partitionBy("cliente").orderBy("fecha")
ventana_acum = Window.partitionBy("cliente").orderBy("fecha").rowsBetween(
    Window.unboundedPreceding, 0
)

(
    df_tend
    .withColumn("monto_anterior",  lag("monto", 1).over(ventana_hist))
    .withColumn("monto_siguiente", lead("monto", 1).over(ventana_hist))
    .withColumn("acumulado",       spark_sum("monto").over(ventana_acum))
    .show()
)

**Resultado esperado:**
```
+-------+----------+------+--------------+---------------+---------+
|cliente|     fecha| monto|monto_anterior|monto_siguiente|acumulado|
+-------+----------+------+--------------+---------------+---------+
|    Ana|2024-01-05| 500.0|          null|         1500.0|    500.0|
|    Ana|2024-01-10|1500.0|         500.0|          200.0|   2000.0|
|    Ana|2024-01-15| 200.0|        1500.0|         2200.0|   2200.0|
|    Ana|2024-01-20|2200.0|         200.0|           null|   4400.0|
+-------+----------+------+--------------+---------------+---------+
```

---
## 4. Manejo de valores nulos

In [ ]:
from pyspark.sql.functions import coalesce, lit

df_nulos = spark.createDataFrame([
    (1, "Ana",   1500.0, "AR"),
    (2, "Luis",   None,  "MX"),
    (3, "Marta", 2200.0,  None),
    (4, None,    None,   None),
], ["id", "cliente", "monto", "pais"])

print("=== dropna (cualquier nulo) ===")
df_nulos.dropna().show()

print("=== dropna (solo si monto es nulo) ===")
df_nulos.dropna(subset=["monto"]).show()

print("=== fillna (rellenar valores por columna) ===")
df_nulos.fillna({"monto": 0.0, "pais": "DESCONOCIDO", "cliente": "SIN_NOMBRE"}).show()

print("=== coalesce (primera columna no nula) ===")
df_nulos.withColumn("monto_seguro", coalesce(col("monto"), lit(0.0))).show()

---
## 5. UDFs — User Defined Functions

> ⚠️ **Usar solo cuando no exista función nativa**. Las UDFs no usan Catalyst Optimizer y son más lentas.

In [ ]:
# ✅ Preferir: when/otherwise (función nativa)
df_montos = spark.createDataFrame(
    [(1, 500.0), (2, 2500.0), (3, 8000.0)],
    ["id", "monto"]
)

print("=== Con función nativa (RECOMENDADO) ===")
df_montos.withColumn("categoria",
    when(col("monto") < 1000, "bajo")
    .when(col("monto") < 5000, "medio")
    .otherwise("alto")
).show()

# ❌ Solo si necesitas lógica que no tiene equivalente nativo
print("=== Con UDF (solo si no hay alternativa nativa) ===")

@udf(StringType())
def clasificar_monto(monto):
    if monto is None:
        return "sin_dato"
    if monto < 1000:
        return "bajo"
    elif monto < 5000:
        return "medio"
    return "alto"

df_montos.withColumn("categoria_udf", clasificar_monto(col("monto"))).show()

---
## 6. Práctica guiada

### Ejercicio 1 — Simple: métricas por cliente

In [ ]:
transacciones = spark.createDataFrame([
    (1, "Ana",   1500.0),
    (2, "Luis",   800.0),
    (3, "Ana",    200.0),
    (4, "Marta", 2200.0),
    (5, "Luis",   300.0),
    (6, "Ana",    900.0),
], ["id", "cliente", "monto"])

resumen = (
    transacciones
    .groupBy("cliente")
    .agg(
        sum("monto").alias("total"),
        count("id").alias("operaciones"),
        avg("monto").alias("promedio"),
    )
    .orderBy(col("total").desc())
)
resumen.show()

**Resultado esperado:**
```
+-------+------+-----------+------------------+
|cliente| total|operaciones|          promedio|
+-------+------+-----------+------------------+
|    Ana|2600.0|          3|866.6666666666666|
|  Marta|2200.0|          1|            2200.0|
|   Luis|1100.0|          2|             550.0|
+-------+------+-----------+------------------+
```

### Ejercicio 2 — Medio: join + ranking global

In [ ]:
clientes = spark.createDataFrame([
    (1, "Ana",   "AR", "VIP"),
    (2, "Luis",  "MX", "regular"),
    (3, "Marta", "AR", "VIP"),
], ["cliente_id", "nombre", "pais", "segmento"])

tx = spark.createDataFrame([
    (101, 1, 1500.0),
    (102, 2,  800.0),
    (103, 1,  200.0),
    (104, 3, 2200.0),
    (105, 2,  300.0),
    (106, 1,  900.0),
], ["tx_id", "cliente_id", "monto"])

resumen_clientes = (
    tx
    .groupBy("cliente_id")
    .agg(sum("monto").alias("total_monto"))
    .join(clientes, "cliente_id", "inner")
    .withColumn("ranking_global",
        rank().over(Window.orderBy(col("total_monto").desc()))
    )
    .select("ranking_global", "nombre", "pais", "segmento", "total_monto")
    .orderBy("ranking_global")
)
resumen_clientes.show()

**Resultado esperado:**
```
+--------------+------+----+--------+-----------+
|ranking_global|nombre|pais|segmento|total_monto|
+--------------+------+----+--------+-----------+
|             1|   Ana|  AR|     VIP|     2600.0|
|             2| Marta|  AR|     VIP|     2200.0|
|             3|  Luis|  MX| regular|     1100.0|
+--------------+------+----+--------+-----------+
```

---
## 7. Preguntas de revisión

1. ¿Cuál es la diferencia entre una agregación (`groupBy`) y una window function?
2. ¿Cuándo usarías `rank` vs `dense_rank`?
3. ¿Por qué `lag` devuelve `null` en la primera fila?
4. ¿Cuándo está justificado usar una UDF?
5. ¿Qué impacto tienen los joins en el rendimiento de Spark?

---
**Próxima unidad:** Optimización y rendimiento en Spark